# 01 Intro to Clinical ML
### ทำนายความเสี่ยงเบาหวานจากข้อมูลผู้ป่วย

Notebook นี้พาคุณสร้างโมเดล Machine Learning ตั้งแต่ต้นจนจบ, โหลดข้อมูล สำรวจ เตรียม train และประเมินผล โดยใช้ชุดข้อมูล diabetes ที่เป็นมาตรฐาน

> 🎯 **เป้าหมาย:** เข้าใจ pipeline ของ ML ทางคลินิก และวัดผลด้วย metric ที่ถูกต้อง

## 1. ติดตั้งและ import ไลบรารี

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

print('พร้อมแล้ว ✅')

พร้อมแล้ว ✅


## 2. โหลดและสำรวจข้อมูล

เราใช้ชุดข้อมูล diabetes แล้วแปลงเป้าหมายให้เป็นปัญหาแบบ classification (เสี่ยงสูง / ไม่สูง) เพื่อสาธิตการทำนายความเสี่ยง

In [ ]:
raw = load_diabetes(as_frame=True)
df = raw.frame
# แปลงเป็นปัญหา classification: เสี่ยงสูง = ค่าเป้าหมายสูงกว่า median
df['high_risk'] = (df['target'] > df['target'].median()).astype(int)
df = df.drop(columns=['target'])
print(df.shape)
df.head()

(442, 11)


## 3. เตรียมข้อมูล (train / valid / test)

การแบ่งข้อมูลอย่างเป็นธรรมคือหัวใจของการประเมินที่เชื่อถือได้ เรากันชุด test ไว้ไม่ให้โมเดลเห็นเลยจนกว่าจะวัดผลครั้งสุดท้าย

In [ ]:
X = df.drop(columns=['high_risk'])
y = df['high_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f'Train: {X_train.shape[0]} ราย | Test: {X_test.shape[0]} ราย')

Train: 353 ราย | Test: 89 ราย


## 4. Train โมเดล

เริ่มจาก Random Forest ซึ่งทำงานดีกับข้อมูลตารางและตีความ feature importance ได้

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
print('Train เสร็จแล้ว 🎉')

Train เสร็จแล้ว 🎉


## 5. ประเมินผล

ในงานคลินิก **AUROC** และ **recall (sensitivity)** สำคัญมาก เพราะการพลาดผู้ป่วยที่เสี่ยงจริงอันตรายกว่าการเตือนเกิน

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(f'AUROC: {roc_auc_score(y_test, proba):.3f}')
print(classification_report(y_test, pred, target_names=['ไม่เสี่ยง', 'เสี่ยงสูง']))

AUROC: 0.812
              precision    recall  f1-score   support

    ไม่เสี่ยง       0.78      0.80      0.79        44
    เสี่ยงสูง       0.80      0.78      0.79        45

    accuracy                           0.79        89


## 6. ตีความโมเดล (Feature Importance)

แพทย์ต้องเข้าใจว่าโมเดลใช้ปัจจัยใดในการตัดสินใจ, นี่คือหัวใจของ [Clinical AI ที่อธิบายได้](../curriculum/health/clinical-ai.html)

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(5)

bmi    0.182
s5     0.165
bp     0.121
s6     0.098
age    0.087
dtype: float64


## สรุปและก้าวต่อไป

- เราสร้าง pipeline ML ทางคลินิกครบวงจร: โหลด → เตรียม → train → ประเมิน → ตีความ
- AUROC ~0.81 เป็นจุดเริ่มที่ดี ลองปรับ feature และโมเดลอื่นดู

**ลองต่อ:** เปลี่ยนเป็น `GradientBoostingClassifier` แล้วเทียบ AUROC, ส่งผลขึ้น [Open Model Board](../platform/open-model-board.html) ได้เลย!